# Analisis Netflix Reviews: EDA, Regex, dan Bag of Words

Notebook ini berisi analisis data review Netflix yang telah diproses sebelumnya. Fokus analisis mencakup Exploratory Data Analysis (EDA), pembersihan teks menggunakan Regular Expression (Regex), serta ekstraksi fitur menggunakan Bag of Words (CountVectorizer) dan TF-IDF.

## 1. Import Library & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from wordcloud import WordCloud

# Load data preprocessed
df = pd.read_csv('../Output/Netflix_Review_Preprocessed.csv')

print(f"Dataset Shape: {df.shape}")
df.head()

## 2. Exploratory Data Analysis (EDA)
Melihat distribusi skor dan statistik dasar dari dataset.

In [ ]:
sns.set_style('whitegrid')

# 1. Distribusi Skor
plt.figure(figsize=(8, 5))
sns.countplot(x='score', data=df, palette='viridis')
plt.title('Distribusi Skor Review Netflix')
plt.xlabel('Skor')
plt.ylabel('Jumlah')
plt.show()

# 2. Jumlah Review berdasarkan App Version (Top 10)
plt.figure(figsize=(10, 6))
df['appVersion'].value_counts().head(10).plot(kind='bar', color='salmon')
plt.title('Top 10 App Versions by Review Count')
plt.xticks(rotation=45)
plt.show()

## 3. Analisis Regex
Regular Expression digunakan untuk membersihkan teks dari noise yang mungkin masih tersisa atau untuk ekstraksi pola tertentu.

In [ ]:
def regex_cleaner(text):
    if not isinstance(text, str):
        return ""
    # 1. Menghapus username mention (@user)
    text = re.sub(r'@[A-Za-z0-9_]+', '', text)
    # 2. Menghapus URL
    text = re.sub(r'https?:\/\/\S+', '', text)
    # 3. Menghapus karakter non-alfabet (angka dan simbol)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    # 4. Menghapus spasi ganda
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

# Contoh penerapan Regex pada kolom 'content' asli
df['regex_cleaned'] = df['content'].apply(regex_cleaner)

print("Perbandingan Teks Asli vs Regex Cleaned:")
df[['content', 'regex_cleaned']].head()

## 3.1. Sentiment Insight via Regex (Keyword Matching)
Di bagian ini, kita menggunakan Regex untuk mencari pola kata kunci yang mengindikasikan sentimen positif atau negatif secara spesifik.

In [ ]:
def get_sentiment_regex(text):
    if not isinstance(text, str): return "Neutral"
    
    # Pola Regex untuk kata-kata Positif
    pos_pattern = r'\b(bagus|puas|mantap|keren|suka|love|best|good|great|enjoy|amazing)\b'
    # Pola Regex untuk kata-kata Negatif
    neg_pattern = r'\b(kecewa|jelek|buruk|bad|worst|disappointed|slow|mahal|error|rugi|benci|hate)\b'
    
    if re.search(pos_pattern, text.lower()):
        return "Positive Insight"
    elif re.search(neg_pattern, text.lower()):
        return "Negative Insight"
    else:
        return "Neutral/Other"

# Terapkan pada kolom regex_cleaned
df['regex_sentiment'] = df['regex_cleaned'].apply(get_sentiment_regex)

print("Hasil Ekstraksi Sentimen menggunakan Regex:")
display(df['regex_sentiment'].value_counts())

# Visualisasi perbandingan skor asli dengan deteksi regex
plt.figure(figsize=(10, 6))
sns.countplot(x='score', hue='regex_sentiment', data=df)
plt.title('Validasi Regex Sentiment vs Skor User')
plt.show()

## 4. Bag of Words (CountVectorizer)

In [ ]:
# Menggunakan kolom 'final_content' yang sudah bersih
data_text = df['final_content'].fillna('')

count_vect = CountVectorizer(max_features=1000)
bow_matrix = count_vect.fit_transform(data_text)

# Visualisasi 20 Kata Terpopuler dalam BoW
sum_words = bow_matrix.sum(axis=0)
words_freq = [(word, sum_words[0, idx]) for word, idx in count_vect.vocabulary_.items()]
words_freq = sorted(words_freq, key = lambda x: x[1], reverse=True)

df_bow = pd.DataFrame(words_freq[:20], columns=['Word', 'Frequency'])

plt.figure(figsize=(12, 6))
sns.barplot(x='Frequency', y='Word', data=df_bow, palette='Blues_r')
plt.title('Top 20 Kata Terpopuler (Bag of Words)')
plt.show()

## 5. TF-IDF Analysis
Menganalisis bobot kepentingan kata menggunakan Term Frequency-Inverse Document Frequency.

In [ ]:
tfidf_vect = TfidfVectorizer(max_features=1000)
tfidf_matrix = tfidf_vect.fit_transform(data_text)

# Mendapatkan skor TF-IDF rata-rata
importance = np.asarray(tfidf_matrix.mean(axis=0)).ravel().tolist()
tfidf_importance = pd.DataFrame({'Word': tfidf_vect.get_feature_names_out(), 'Importance': importance})
tfidf_importance = tfidf_importance.sort_values(by='Importance', ascending=False).head(20)

plt.figure(figsize=(12, 6))
sns.barplot(x='Importance', y='Word', data=tfidf_importance, palette='Greens_r')
plt.title('Top 20 Kata Berdasarkan Bobot TF-IDF')
plt.show()

## 6. WordCloud Visualization
Visualisasi kata-kata yang paling sering muncul secara intuitif.

In [ ]:
all_words = ' '.join(data_text)
wordcloud = WordCloud(width=800, height=400, background_color='black', colormap='Spectral').generate(all_words)

plt.figure(figsize=(15, 8))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Netflix Review WordCloud')
plt.show()

## 7. Kesimpulan

1. **EDA**: Distribusi skor menunjukkan sentimen pengguna terhadap aplikasi Netflix. Versi aplikasi tertentu mungkin memiliki jumlah keluhan atau pujian yang lebih banyak.
2. **Regex**: Sangat efektif untuk membersihkan sisa noise seperti mention, URL, atau karakter non-standar yang mengganggu proses modeling.
3. **Bag of Words & TF-IDF**: Memberikan gambaran kata kunci utama pengguna. BoW fokus pada frekuensi, sementara TF-IDF memberikan bobot lebih pada kata-kata yang lebih informatif/spesifik bagi dokumen tertentu.